# 02. 2D Pose Estimate와 Nav Goal

## 이 Notebook으로 하는 일

이미 실행 중인 **수동 navigation lane**에 AMCL 초기 pose를 보내고 `/move_base` action으로 목적지를 전송합니다. RViz의 `2D Pose Estimate`와 `2D Nav Goal`에 해당하지만 action client를 사용해 성공·실패·취소 상태를 확인합니다.

## 셀 실행 방법

회색 code cell은 **Python 코드**입니다. 일반 terminal에 붙여 넣지 말고 Jupyter에서 `Shift+Enter`로 한 셀씩 실행하십시오. 각 셀은 앞 셀의 `CONFIG`, `ROS_ENV`, ROS publisher/client를 사용하므로 중간 셀부터 실행하지 않습니다. 오류가 나오면 다음 셀로 넘어가지 말고 원인을 수정한 뒤 실패한 셀부터 다시 실행합니다.

## 시작 전 필수 조건

1. [`notebooks/README.md`](README.md)의 수동 navigation 순서대로 별도 terminal의 sensor, navigation, robot tunnel/supervisor를 시작합니다.
2. Jupyter를 시작한 terminal에서 `bash notebooks/check_environment.sh navigation`을 실행해 `[FAIL]`이 없는지 확인합니다.
3. `./run.sh` managed system이나 mapping Notebook과 함께 사용하지 않습니다. code cell도 충돌 node를 다시 검사합니다.
4. 로봇을 정지시키고 물리적 비상 정지를 손에 둡니다.
5. 첫 code cell의 `INITIAL_X/Y/YAW`는 현재 위치의 승인된 map 좌표, `GOAL_X/Y/YAW`는 목적지의 승인된 map 좌표인지 확인합니다. 기본 goal은 3F 계단 시작점 commissioning 값입니다.
6. `ARM_MOTION`은 localization과 경로를 확인할 때까지 `False`로 둡니다.

이 Notebook은 navigation process를 시작하거나 robot tunnel을 만들지 않습니다. readiness 실패 시 [`docs/ROSBAG-RECORD`](../docs/ROSBAG-RECORD)의 해당 terminal을 수정한 뒤 검사부터 다시 실행합니다.

In [ ]:
from pathlib import Path
import math
import os
import subprocess
import sys
import time

WORKSPACE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG_PATH = WORKSPACE / "config.env"
ROS_SETUP = Path("/opt/ros/noetic/setup.bash")
DEVEL_SETUP = WORKSPACE / "devel/setup.bash"

if not CONFIG_PATH.is_file() or not ROS_SETUP.is_file() or not DEVEL_SETUP.is_file():
    raise RuntimeError("workspace, config.env, or ROS Noetic setup is missing")

def read_config(path):
    values = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if line and not line.startswith("#"):
            key, value = line.split("=", 1)
            values[key] = value
    return values

def sourced_environment():
    command = f"source {ROS_SETUP} && source {DEVEL_SETUP} && env -0"
    completed = subprocess.run(["bash", "-lc", command], check=True, stdout=subprocess.PIPE)
    environment = os.environ.copy()
    for entry in completed.stdout.split(b"\0"):
        if entry and b"=" in entry:
            key, value = entry.split(b"=", 1)
            environment[key.decode()] = value.decode()
    return environment

CONFIG = read_config(CONFIG_PATH)
ROS_ENV = sourced_environment()
ROS_ENV["ROS_MASTER_URI"] = f"http://{CONFIG['ROS_MASTER_HOST']}:{CONFIG['ROS_MASTER_PORT']}"
route = subprocess.run(["ip", "-4", "route", "get", CONFIG["ROS_MASTER_HOST"]], check=True, text=True, stdout=subprocess.PIPE).stdout.split()
ROS_ENV["ROS_IP"] = route[route.index("src") + 1]
ROS_ENV.pop("ROS_HOSTNAME", None)
os.environ.update(ROS_ENV)
for path in (Path("/opt/ros/noetic/lib/python3/dist-packages"), WORKSPACE / "devel/lib/python3/dist-packages"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

INITIAL_X = float(CONFIG["NAV_START_X"])
INITIAL_Y = float(CONFIG["NAV_START_Y"])
INITIAL_YAW = float(CONFIG["NAV_START_YAW"])
GOAL_X = 10.054808
GOAL_Y = -6.652934
GOAL_YAW = 1.395796
GOAL_TIMEOUT_SEC = 180.0
ARM_MOTION = False
MOVE_BASE_CLIENT = None
print(f"ROS_MASTER_URI={ROS_ENV['ROS_MASTER_URI']}")

## Gate 1: ROS graph readiness

이 셀은 읽기 전용 검사입니다. 필수 node와 topic type을 확인하고 managed mission node가 있으면 거부합니다. `/stair_supervisor/state`가 NAV(1), connected=True여야 move_base의 `/navigation/cmd_vel`이 실제 로봇으로 전달됩니다. 성공하면 supervisor message가 출력됩니다. 실패하면 다음 셀로 넘어가지 말고 외부 terminal을 복구한 뒤 이 셀을 다시 실행합니다.

In [ ]:
def ros_output(argv, timeout=10):
    return subprocess.run(argv, env=ROS_ENV, check=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout).stdout.strip()

nodes = set(ros_output(["rosnode", "list"]).splitlines())
managed = {"/mission_manager", "/multifloor_manager"} & nodes
if managed:
    raise RuntimeError(f"managed mission mode와 direct navigation을 섞을 수 없습니다: {sorted(managed)}")
required_nodes = {"/map_server", "/amcl", "/move_base", "/stair_supervisor"}
missing_nodes = required_nodes - nodes
if missing_nodes:
    raise RuntimeError(f"docs/ROSBAG-RECORD의 수동 navigation 단계를 먼저 실행하십시오: {sorted(missing_nodes)}")
required_topics = {
    "/scan": "sensor_msgs/LaserScan",
    "/tron/wheel_odom_raw": "nav_msgs/Odometry",
    "/map": "nav_msgs/OccupancyGrid",
    "/amcl_pose": "geometry_msgs/PoseWithCovarianceStamped",
    "/stair_supervisor/state": "stair_supervisor/SupervisorState",
}
for topic, expected in required_topics.items():
    actual = ros_output(["rostopic", "type", topic], timeout=3)
    if actual != expected:
        raise RuntimeError(f"topic type mismatch: {topic} expected={expected} actual={actual}")
print(ros_output(["rostopic", "echo", "-n", "1", "/stair_supervisor/state"], timeout=10))

## Gate 2: AMCL 초기 pose와 localization

로봇이 정지한 상태에서 `INITIAL_X/Y/YAW`가 현재 실제 위치와 같은지 마지막으로 확인한 뒤 실행합니다. 이 셀은 `/initialpose`를 실제로 세 번 publish하지만 로봇을 움직이지는 않습니다. 다음 localization 셀에서 AMCL 위치와 covariance를 확인하고 값이 틀리거나 계속 크면 goal 셀로 넘어가지 말고 초기 pose부터 다시 설정합니다.

In [ ]:
import rospy
from geometry_msgs.msg import PoseWithCovarianceStamped
from std_srvs.srv import Empty

if not rospy.core.is_initialized():
    rospy.init_node("tron1_pose_goal_notebook", anonymous=True, disable_signals=True)

pose = PoseWithCovarianceStamped()
pose.header.frame_id = "map"
pose.pose.pose.position.x = INITIAL_X
pose.pose.pose.position.y = INITIAL_Y
pose.pose.pose.orientation.z = math.sin(INITIAL_YAW / 2.0)
pose.pose.pose.orientation.w = math.cos(INITIAL_YAW / 2.0)
pose.pose.covariance[0] = 0.01
pose.pose.covariance[7] = 0.01
pose.pose.covariance[35] = 0.01
publisher = rospy.Publisher("/initialpose", PoseWithCovarianceStamped, queue_size=1, latch=True)
deadline = time.monotonic() + 5.0
while publisher.get_num_connections() == 0 and time.monotonic() < deadline:
    rospy.sleep(0.1)
if publisher.get_num_connections() == 0:
    raise RuntimeError("AMCL is not subscribed to /initialpose")
for _ in range(3):
    pose.header.stamp = rospy.Time.now()
    publisher.publish(pose)
    rospy.sleep(0.2)
print(f"initial pose published: x={INITIAL_X} y={INITIAL_Y} yaw={INITIAL_YAW}")

## localization 확인

이 셀은 managed runtime과 같은 production readiness 기준을 사용합니다. 로봇이 정지해 있어도 각 sample 전에 `/request_nomotion_update`를 호출하고, 호출 전 기준보다 새 timestamp인 AMCL message만 받습니다. 연속 3개 message의 covariance는 `x <= 0.05`, `y <= 0.05`, `yaw <= 0.10`이어야 합니다. service가 없거나 새 sample이 오지 않거나 covariance가 하나라도 넘으면 명확한 오류로 중단합니다.

코드는 마지막 pose와 요청한 initial pose의 위치·yaw 차이도 출력하지만, 실제 로봇 위치가 맞는지는 운영자가 직접 확인해야 합니다. `LOCALIZATION GATE: PASS`와 실제 위치 일치를 모두 확인하기 전에는 motion gate로 넘어가지 않습니다.

In [ ]:
from multifloor_manager.readiness import DEFAULT_READINESS_POLICY

policy = DEFAULT_READINESS_POLICY
samples = []
try:
    rospy.wait_for_service("/request_nomotion_update", timeout=2.0)
except rospy.ROSException as error:
    raise RuntimeError("AMCL nomotion update service is unavailable") from error
request_nomotion_update = rospy.ServiceProxy("/request_nomotion_update", Empty)
try:
    baseline = rospy.wait_for_message("/amcl_pose", PoseWithCovarianceStamped, timeout=10.0)
except rospy.ROSException as error:
    raise RuntimeError("AMCL pose baseline is unavailable") from error
previous_stamp_ns = baseline.header.stamp.to_nsec()
for sample_index in range(policy.required_pose_samples):
    try:
        request_nomotion_update()
    except rospy.ServiceException as error:
        raise RuntimeError(f"AMCL nomotion update failed: sample={sample_index + 1}") from error
    sample_deadline = time.monotonic() + 10.0
    while True:
        remaining = sample_deadline - time.monotonic()
        if remaining <= 0.0:
            raise RuntimeError(f"AMCL sample did not advance after nomotion update: sample={sample_index + 1}")
        try:
            message = rospy.wait_for_message("/amcl_pose", PoseWithCovarianceStamped, timeout=min(1.0, remaining))
        except rospy.ROSException as error:
            if time.monotonic() >= sample_deadline:
                raise RuntimeError(f"AMCL sample did not arrive after nomotion update: sample={sample_index + 1}") from error
            continue
        stamp_ns = message.header.stamp.to_nsec()
        if stamp_ns > previous_stamp_ns:
            break
        rospy.sleep(0.05)
    covariance = (message.pose.covariance[0], message.pose.covariance[7], message.pose.covariance[35])
    limits = (policy.max_covariance_x, policy.max_covariance_y, policy.max_covariance_yaw)
    if any(value > limit for value, limit in zip(covariance, limits)):
        raise RuntimeError(f"AMCL covariance gate failed: actual={covariance} limits={limits}")
    samples.append(message)
    previous_stamp_ns = stamp_ns

amcl_message = samples[-1]
amcl_x = amcl_message.pose.pose.position.x
amcl_y = amcl_message.pose.pose.position.y
orientation = amcl_message.pose.pose.orientation
amcl_yaw = 2.0 * math.atan2(orientation.z, orientation.w)
position_error = math.hypot(amcl_x - INITIAL_X, amcl_y - INITIAL_Y)
yaw_error = math.atan2(math.sin(amcl_yaw - INITIAL_YAW), math.cos(amcl_yaw - INITIAL_YAW))
print(f"AMCL x={amcl_x:.3f} y={amcl_y:.3f} yaw={amcl_yaw:.3f}")
print(f"difference from requested initial pose: position={position_error:.3f}m yaw={yaw_error:.3f}rad")
print(f"covariance xx={amcl_message.pose.covariance[0]:.4f} yy={amcl_message.pose.covariance[7]:.4f} yaw={amcl_message.pose.covariance[35]:.4f}")
print("LOCALIZATION GATE: PASS")

## Gate 3: Nav Goal 전송

아래 항목을 모두 직접 확인한 뒤에만 바로 아래 arming 셀의 `ARM_MOTION = True`를 적용하고 그 다음 goal 셀을 실행합니다.

- 로봇이 올바른 map과 floor에 있다.
- 직전 셀에 `LOCALIZATION GATE: PASS`가 출력됐고 AMCL pose가 현재 실제 위치와 맞는다.
- `GOAL_X/Y/YAW`가 이 map에서 승인된 목적지다.
- 예상 경로에 사람과 장애물이 없고 물리적 비상 정지를 즉시 사용할 수 있다.
- supervisor가 NAV(1), connected=True다.

이 셀은 실제 `/move_base` action goal을 보내므로 새 좌표 탐색용으로 사용하지 않습니다. 완료까지 기다리고, timeout이면 같은 goal을 자동 취소합니다. 성공 기준은 `navigation succeeded` 출력입니다.

In [ ]:
ARM_MOTION = False  # 위 checklist를 모두 확인한 뒤에만 True
print(f"motion armed={ARM_MOTION}")

In [ ]:
import actionlib
from actionlib_msgs.msg import GoalStatus
from move_base_msgs.msg import MoveBaseAction, MoveBaseGoal
from stair_supervisor.msg import SupervisorState

if not ARM_MOTION:
    raise RuntimeError("motion is disarmed; verify the site and set ARM_MOTION = True")
supervisor = rospy.wait_for_message("/stair_supervisor/state", SupervisorState, timeout=5.0)
if supervisor.state != 1 or not supervisor.connected:
    raise RuntimeError(f"stair supervisor must be NAV(1) and connected: {supervisor}")
MOVE_BASE_CLIENT = actionlib.SimpleActionClient("/move_base", MoveBaseAction)
if not MOVE_BASE_CLIENT.wait_for_server(rospy.Duration(10.0)):
    raise RuntimeError("/move_base action server is unavailable")
goal = MoveBaseGoal()
goal.target_pose.header.frame_id = "map"
goal.target_pose.header.stamp = rospy.Time.now()
goal.target_pose.pose.position.x = GOAL_X
goal.target_pose.pose.position.y = GOAL_Y
goal.target_pose.pose.orientation.z = math.sin(GOAL_YAW / 2.0)
goal.target_pose.pose.orientation.w = math.cos(GOAL_YAW / 2.0)
MOVE_BASE_CLIENT.send_goal(goal)
finished = MOVE_BASE_CLIENT.wait_for_result(rospy.Duration(GOAL_TIMEOUT_SEC))
if not finished:
    MOVE_BASE_CLIENT.cancel_goal()
    raise RuntimeError(f"navigation timed out after {GOAL_TIMEOUT_SEC}s and was cancelled")
state = MOVE_BASE_CLIENT.get_state()
if state != GoalStatus.SUCCEEDED:
    raise RuntimeError(f"navigation failed: state={state} text={MOVE_BASE_CLIENT.get_goal_status_text()}")
print(f"navigation succeeded: x={GOAL_X} y={GOAL_Y} yaw={GOAL_YAW}")

## 긴급 goal 취소와 종료

위 goal 셀이 기다리는 중이면 Jupyter 메뉴의 **Kernel → Interrupt Kernel**을 누른 뒤 아래 셀을 실행합니다. 이 셀은 action client가 있으면 현재 goal만 취소하고, client가 없으면 `/move_base/cancel`로 모든 move_base goal 취소를 요청합니다. Kernel이 응답하지 않거나 로봇이 안전하지 않으면 Notebook보다 물리적 비상 정지를 먼저 사용하십시오.

취소 또는 성공 후 navigation, supervisor, tunnel은 이 Notebook 소유가 아니므로 계속 실행됩니다. 작업을 끝낼 때는 [`docs/ROSBAG-RECORD`](../docs/ROSBAG-RECORD)의 역순으로 각 소유 terminal에서 `Ctrl+C`를 누릅니다.

In [ ]:
if MOVE_BASE_CLIENT is not None:
    MOVE_BASE_CLIENT.cancel_goal()
    print("move_base goal cancel requested")
else:
    subprocess.run(["rostopic", "pub", "-1", "/move_base/cancel", "actionlib_msgs/GoalID", "{}"], env=ROS_ENV, check=True, timeout=5)
    print("move_base cancel topic published")